# Lab 6: Decision Tree Classification

**Objective:** Train and visualize a Decision Tree on the Wine Quality dataset; study the effect of `max_depth` and splitting `criterion` (gini vs entropy). Target: `quality`.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

In [ ]:
zip_path = Path("winequality.zip")
csv_path = Path("winequality-red.csv")

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_file:
        csv_files = [n for n in zip_file.namelist() if n.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was found inside winequality.zip.")
        csv_bytes = zip_file.read(csv_files[0])
        data = pd.read_csv(BytesIO(csv_bytes), sep=None, engine="python")
elif csv_path.exists():
    data = pd.read_csv(csv_path, sep=None, engine="python")
else:
    raise FileNotFoundError("Put winequality.zip or winequality-red.csv in the same folder.")

print("Dataset shape:", data.shape)
data.head()

In [ ]:
print(data.columns.tolist())
data.info()
print(data.isnull().sum())
print(data["quality"].value_counts().sort_index())

In [ ]:
class_counts = data["quality"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Distribution of Wine Quality Classes")
plt.xlabel("Wine quality"); plt.ylabel("Number of samples")
plt.tight_layout(); plt.show()

In [ ]:
X = data.drop(columns=["quality"])
y = data["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
depth_values = [1, 2, 3, 4, 5, 6, 8, 10, None]
criteria = ["gini", "entropy"]

results = []
trained_models = {}

for criterion in criteria:
    for depth in depth_values:
        model = DecisionTreeClassifier(criterion=criterion, max_depth=depth, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)

        train_predictions = model.predict(X_train)
        test_predictions = model.predict(X_test)

        results.append({
            "criterion": criterion,
            "max_depth": depth,
            "train_accuracy": accuracy_score(y_train, train_predictions),
            "test_accuracy": accuracy_score(y_test, test_predictions),
            "precision_weighted": precision_score(y_test, test_predictions, average="weighted", zero_division=0),
            "recall_weighted": recall_score(y_test, test_predictions, average="weighted", zero_division=0),
            "f1_weighted": f1_score(y_test, test_predictions, average="weighted", zero_division=0),
            "actual_depth": model.get_depth(),
            "number_of_leaves": model.get_n_leaves(),
        })
        trained_models[(criterion, depth)] = model

results_df = pd.DataFrame(results)
display(results_df.round(3))

In [ ]:
plt.figure(figsize=(8, 5))
for criterion in criteria:
    subset = results_df[(results_df["criterion"] == criterion) &
                        (results_df["max_depth"].notna())].sort_values("max_depth")
    plt.plot(subset["max_depth"], subset["test_accuracy"], marker="o", label=criterion)

plt.title("Effect of Maximum Tree Depth on Test Accuracy")
plt.xlabel("Maximum tree depth"); plt.ylabel("Test accuracy")
plt.xticks([1, 2, 3, 4, 5, 6, 8, 10])
plt.grid(True); plt.legend(title="Splitting criterion")
plt.tight_layout(); plt.show()

In [ ]:
summary_rows = []
for criterion in criteria:
    cr = results_df[results_df["criterion"] == criterion]
    best_row = cr.loc[cr["f1_weighted"].idxmax()]
    summary_rows.append({
        "criterion": criterion,
        "best_max_depth": best_row["max_depth"],
        "test_accuracy": best_row["test_accuracy"],
        "weighted_precision": best_row["precision_weighted"],
        "weighted_recall": best_row["recall_weighted"],
        "weighted_f1": best_row["f1_weighted"],
    })

display(pd.DataFrame(summary_rows).round(3))

best_accuracy_row = results_df.loc[results_df["test_accuracy"].idxmax()]
print("Highest test accuracy:", best_accuracy_row["criterion"],
      best_accuracy_row["max_depth"], round(best_accuracy_row["test_accuracy"], 3))

In [ ]:
# Visualize a small tree
model = trained_models[("gini", 3)]
plt.figure(figsize=(18, 8))
plot_tree(model, feature_names=X.columns, filled=True, fontsize=8)
plt.title("Decision Tree (gini, max_depth=3)")
plt.show()

## Conclusion
- Training accuracy rises to 1.0 with unlimited depth while test accuracy stays ~0.59-0.61 → overfitting.
- Best test accuracy: entropy, max_depth=10 (0.609). Gini and entropy perform similarly (weighted F1 ≈ 0.607).
- Results are from a single split, so small differences aren't conclusive.